# Comparative Genomics & Phylogeny of the Périgord Black Truffle (*Tuber melanosporum*)

1. **Comparative genomics** from literature-sourced values (report Tables 1–2, Figures 1–5)
2. **NCBI Assembly cross-check** of genome sizes, plus *T. brumale* and *T. indicum*
3. **De novo ITS phylogeny**: NCBI GenBank → MAFFT → trimming → IQ-TREE 2 (ModelFinder + 1000 ultrafast bootstraps)

**Runtime → Run all.** Takes about 3–5 minutes on Colab.

In [ ]:
# Colab only: get the code and install the phylogenetics tools
!git clone -q https://github.com/moudi-eng/tuber-comparative-genomics.git
%cd tuber-comparative-genomics
!apt-get -qq install -y mafft iqtree > /dev/null
!pip install -q -r requirements.txt
!mafft --version; (iqtree2 --version || iqtree --version) | head -1

In [ ]:
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')  # run from the repository root when used locally

EMAIL = 'your.name@example.com'   # NCBI asks for a contact e-mail; put yours here

import json, pandas as pd
from IPython.display import Image, display

def show(*figs, run='tuber', width=650):
    for f in figs:
        display(Image(f'results/{run}/figures/{f}.png', width=width))

## 1. Comparative genomics (literature-sourced)

In [ ]:
!python src/pipeline.py --config configs/tuber.json --steps comparative
display(pd.read_csv('results/tuber/table1_genomic_features.csv'))
display(pd.read_csv('results/tuber/table2_divergence_ages.csv'))
show('fig1_genome_size', 'fig2_size_vs_genes', 'fig3_te_content', 'fig4_cazyme_range', 'fig5_published_cladogram')

## 2. Cross-check genome sizes against NCBI Assembly

In [ ]:
!python src/pipeline.py --config configs/tuber.json --steps ncbi --email {EMAIL}
display(pd.read_csv('results/tuber/ncbi_assembly_crosscheck.csv'))
show('fig6_size_vs_genes_extended')

## 3. De novo ITS phylogeny (MAFFT + IQ-TREE)

Downloads 3 ITS sequences per species from GenBank (outgroup: *Choiromyces venosus*), aligns them, removes gap-rich columns and infers a maximum-likelihood tree. The result is compared with the published cladogram (Robinson–Foulds distance).

In [ ]:
!python src/pipeline.py --config configs/tuber.json --steps phylogeny --email {EMAIL}
display(pd.read_csv('results/tuber/sequences_used.csv')[['species', 'accession', 'length', 'description']])
show('fig7_ITS_tree', 'fig8_distance_heatmap', width=750)

In [ ]:
summary = json.load(open('results/tuber/summary.json'))
print(json.dumps(summary['phylogeny'], indent=2))

## 4. Try another marker or other species

Markers: `ITS`, `LSU`, `RPB2`, `EF1a`. To change species, edit `configs/tuber.json` or copy it.

In [ ]:
MARKER = 'LSU'   # change me
!python src/pipeline.py --config configs/tuber.json --steps phylogeny --marker {MARKER} --name tuber_{MARKER} --email {EMAIL}
show(f'fig7_{MARKER}_tree', run=f'tuber_{MARKER}', width=750)